#### EDA 08: Geolocation table
**File:** `olist_geolocation_dataset.csv`  |  **Goal:** understand the zip prefix to coordinates lookup, and what has to be cleaned before it is used. This table only. No joins happen here. Cross-table checks (does every customer and seller prefix exist here?) are listed at the end.

**Flow:** Grain → Duplicates → Rows per prefix → Zip format → Coordinates → Spread inside a prefix → City and state text → Prefix lookup → Summary

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data

geo_raw = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
geo = geo_raw.copy()                              # all changes happen on the copy, raw stays untouched

print("shape:", geo.shape)
print(geo.dtypes)
geo.head(3)

shape: (1000163, 5)
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


- 1,000,163 rows and 5 columns: zip prefix, latitude, longitude, city and state. It is the largest table, with about 10 times the rows of orders.
- The zip prefix is stored as a number.
- One row is one coordinate point, not one zip prefix. The table is a point cloud per prefix.

#### Missing values and duplicates
**Question:** are there gaps, and how many rows are repeats?

In [2]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(geo),
                        "n_missing": geo.isna().sum(),
                        "pct_missing": (geo.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

dups = geo.duplicated().sum()
print("fully duplicated rows            :", dups, f"({dups / len(geo) * 100:.1f}%)")
print("distinct rows                    :", len(geo.drop_duplicates()))
print("repeated (prefix, lat, lng)      :", geo.duplicated(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]).sum())
print("distinct (prefix, lat, lng)      :", len(geo.drop_duplicates(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])))

,n_rows,n_missing,pct_missing
geolocation_zip_code_prefix,1000163,0,0.0
geolocation_lat,1000163,0,0.0
geolocation_lng,1000163,0,0.0
geolocation_city,1000163,0,0.0
geolocation_state,1000163,0,0.0


fully duplicated rows            : 261831 (26.2%)


distinct rows                    : 738332
repeated (prefix, lat, lng)      : 280009


distinct (prefix, lat, lng)      : 720154


- No column has a missing value.
- 261,831 rows (26.2%) are exact copies of another row. 738,332 rows are distinct.
- 280,009 rows repeat an earlier (prefix, latitude, longitude). This includes the exact copies, and 18,178 of them differ only in city or state text. Only 720,154 distinct points remain.
- **Decision:** repeated rows add weight and no information. Remove exact copies before any statistic is computed. The raw frame stays as loaded.

In [3]:
geo = geo.drop_duplicates().reset_index(drop=True)      # working copy without exact copies
print("rows after removing exact copies:", len(geo))

rows after removing exact copies: 738332


#### Rows per zip prefix
**Question:** how many points does each prefix have?

In [4]:
rows_per_prefix = geo_raw.groupby("geolocation_zip_code_prefix").size()          # on the raw table, to show the original shape
bins = pd.cut(rows_per_prefix, [0, 1, 5, 20, 100, 1000, np.inf], labels=["1", "2-5", "6-20", "21-100", "101-1000", "1000+"])
dist = bins.value_counts().sort_index().rename("n_prefixes").to_frame()
dist["pct"] = (dist["n_prefixes"] / len(rows_per_prefix) * 100).round(1)
display(dist)

print("distinct zip prefixes      :", len(rows_per_prefix))
print("rows per prefix: median", rows_per_prefix.median(), "| mean", round(rows_per_prefix.mean(), 1), "| max", rows_per_prefix.max(),
      "(prefix", rows_per_prefix.idxmax(), ")")

,n_prefixes,pct
1,1043,5.5
2-5,2021,10.6
6-20,4473,23.5
21-100,8753,46.0
101-1000,2723,14.3
1000+,2,0.0


distinct zip prefixes      : 19015
rows per prefix: median 29.0 | mean 52.6 | max 1146 (prefix 24220 )


- 19,015 distinct zip prefixes. The median prefix has 29 points, the mean is 52.6 and the maximum is 1,146 (prefix 24220).
- Most prefixes have 21 to 100 points. 1,043 prefixes (5.5%) have a single point.
- Each prefix is a cloud of points, so one number per prefix must be chosen. This is done in the lookup section.

#### Zip prefix format
**Question:** does the prefix lose its leading zero here too?

In [5]:
zip_len = geo["geolocation_zip_code_prefix"].astype(str).str.len().value_counts().sort_index()
print("zip prefix length (digits), after removing copies:")
print(zip_len.to_string())

geo["zip5"] = geo["geolocation_zip_code_prefix"].astype(str).str.zfill(5)
print("\nlength of the raw prefix (digits), original rows:")
print(geo_raw["geolocation_zip_code_prefix"].astype(str).str.len().value_counts().sort_index().to_string())

zip prefix length (digits), after removing copies:
geolocation_zip_code_prefix
4    158832
5    579500



length of the raw prefix (digits), original rows:

geolocation_zip_code_prefix
4    245733
5    754430


- Same pattern as customers and sellers: 245,733 of the original rows (24.6%) have a 4-digit prefix because the leading zero was lost.
- The prefix is a number in all three tables, so a numeric join works. `zip5` is added as text for display only.

#### Coordinates
**Question:** are the coordinates inside Brazil?

In [6]:
display(geo[["geolocation_lat", "geolocation_lng"]].describe(percentiles=[0.01, 0.5, 0.99]).round(2))

# A box around Brazil (a rough check, not exact borders)
BR_LAT, BR_LNG = (-33.75, 5.27), (-73.99, -34.79)
in_brazil = geo["geolocation_lat"].between(*BR_LAT) & geo["geolocation_lng"].between(*BR_LNG)

outside = geo[~in_brazil]
print("rows outside the Brazil box    :", len(outside), f"({(~in_brazil).mean() * 100:.3f}%)")
print("distinct prefixes affected     :", outside["geolocation_zip_code_prefix"].nunique())
print("prefixes with NO point inside the box:",
      (geo.assign(ok=in_brazil).groupby("geolocation_zip_code_prefix")["ok"].sum() == 0).sum())
outside.head(5)

,geolocation_lat,geolocation_lng
count,738332.00,738332.00
mean,-21.00,-46.46
std,5.89,4.39
min,-36.61,-101.47
1%,-30.12,-57.69
50%,-22.87,-46.65
99%,-2.44,-34.93
max,45.07,121.11


rows outside the Brazil box    : 33 (0.004%)
distinct prefixes affected     : 21
prefixes with NO point inside the box: 5


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,zip5
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP,18243
355591,28165,41.614052,-8.411675,vila nova de campos,RJ,28165
355603,28155,-34.586422,-58.732101,santa maria,RJ,28155
355694,28155,42.439286,13.820214,santa maria,RJ,28155
356233,28333,38.381672,-6.328200,raposo,RJ,28333


- Latitude runs from -36.61 to 45.07 and longitude from -101.47 to 121.11. Brazil only covers about -33.75 to 5.27 and -73.99 to -34.79.
- Only 33 distinct rows (0.004%) fall outside the box, in 21 prefixes. The first five are listed above, with coordinates in Europe, the Atlantic and Argentina but a Brazilian state.
- 5 prefixes have no valid point at all, so they cannot be located.
- **Decision:** for the lookup, ignore points outside Brazil. They are wrong, and they are few (0.004% of rows).

#### Spread of points inside a prefix
**Question:** how far apart are the points of one prefix? A wide spread means the prefix has no single reliable location.

In [7]:
valid = geo[in_brazil]

box = valid.groupby("geolocation_zip_code_prefix").agg(
    n_points=("geolocation_lat", "size"),
    lat_min=("geolocation_lat", "min"), lat_max=("geolocation_lat", "max"),
    lng_min=("geolocation_lng", "min"), lng_max=("geolocation_lng", "max"))

# Rough size of the box of points in km (1 degree of latitude is about 111 km)
mid_lat = (box["lat_min"] + box["lat_max"]) / 2
box["spread_km"] = np.sqrt(((box["lat_max"] - box["lat_min"]) * 111) ** 2 +
                           ((box["lng_max"] - box["lng_min"]) * 111 * np.cos(np.radians(mid_lat))) ** 2)

multi = box[box["n_points"] > 1]
display(multi["spread_km"].describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame("spread_km"))
print("prefixes with more than 1 point:", len(multi))
print("spread above 50 km  :", (box["spread_km"] > 50).sum())
print("spread above 200 km :", (box["spread_km"] > 200).sum())
print("spread above 1000 km:", (box["spread_km"] > 1000).sum())

,spread_km
count,17819.0
mean,17.3
std,126.5
min,0.0
50%,2.8
90%,18.2
99%,288.9
max,3825.2


prefixes with more than 1 point: 17819
spread above 50 km  : 484
spread above 200 km : 246
spread above 1000 km: 53


- For the 17,819 prefixes with more than one point, the median spread is 2.8 km and 90% are within 18.2 km. That fits a zip prefix of a few neighbourhoods.
- The tail is heavy: 484 prefixes spread over more than 50 km, 246 over more than 200 km, and 53 over more than 1,000 km (the maximum is about 3,825 km).
- A spread of 1,000 km inside a 5-digit prefix is impossible. These prefixes hold points from different places, so some coordinates are wrong even inside the Brazil box.
- **Decision:** summarise each prefix with the median latitude and longitude, which is robust against a few bad points, and flag prefixes with a spread above 50 km as low quality. Do not use the mean.

#### City and state text
**Question:** are the city and state text consistent and clean?

In [8]:
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)).lower().strip()

city = geo["geolocation_city"]
print("rows with non-ASCII city text         :", (~city.map(str.isascii)).sum(), f"({(~city.map(str.isascii)).mean() * 100:.1f}%)")
print("distinct city names raw / no accents  :", city.nunique(), "/", city.map(strip_accents).nunique())
print("city names with symbols, digits or encoding:", city[city.str.contains(r"[\d@/\\()%]")].nunique())
print(city[city.str.contains(r"%")].drop_duplicates().tolist())

# How is Sao Paulo written?
print("\nwriting of Sao Paulo:")
display(city[city.map(strip_accents) == "sao paulo"].value_counts().rename("n_rows").to_frame())

by_prefix = geo.groupby("geolocation_zip_code_prefix").agg(n_cities=("geolocation_city", "nunique"), n_states=("geolocation_state", "nunique"))
print("prefixes with more than one city name:", (by_prefix["n_cities"] > 1).sum(), f"({(by_prefix['n_cities'] > 1).mean() * 100:.1f}%)")
print("prefixes with more than one state    :", (by_prefix["n_states"] > 1).sum())
print("state codes outside the 27 valid ones:", sorted(set(geo["geolocation_state"]) - set(
    "AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split())))

state_share = (geo_raw["geolocation_state"].value_counts(normalize=True) * 100).round(1)
print("\nshare of original rows, top 5 states:", state_share.head(5).to_dict())
conflict = by_prefix.index[by_prefix["n_states"] > 1]
geo[geo["geolocation_zip_code_prefix"].isin(conflict)].groupby("geolocation_zip_code_prefix")["geolocation_state"].unique().head(5).to_frame("states")

rows with non-ASCII city text         : 63436 (8.6%)


distinct city names raw / no accents  : 8011 / 5967
city names with symbols, digits or encoding: 19
['são joão do pau d%26apos%3balho', 'lambari d%26apos%3boeste']

writing of Sao Paulo:


,n_rows
geolocation_city,
sao paulo,79927
são paulo,19718


prefixes with more than one city name: 8556 (45.0%)
prefixes with more than one state    : 8
state codes outside the 27 valid ones: []

share of original rows, top 5 states: {'SP': 40.4, 'MG': 12.6, 'RJ': 12.1, 'RS': 6.2, 'PR': 5.8}


,states
geolocation_zip_code_prefix,
2116,"[SP, RN]"
4011,"[SP, AC]"
21550,"[RJ, AC]"
23056,"[RJ, AC]"
72915,"[GO, DF]"


- City text is messy. 63,436 rows (8.6%, after removing copies) have accents, and the same city appears in several spellings: São Paulo is `sao paulo` in 79,927 rows and `são paulo` in 19,718 rows.
- Removing accents cuts the distinct city names from 8,011 to 5,967.
- 19 city names contain symbols, digits or brackets, and 2 contain web-encoded characters (for example `são joão do pau d%26apos%3balho`), which is a broken text encoding.
- 8,556 prefixes (45.0%) have more than one city name. This comes from spelling variants, so the city text is not usable as a key.
- 8 prefixes appear under two states (for example 4011 as SP and AC, 21550 as RJ and AC). A prefix has one state, so some rows carry a wrong state.
- State codes are all valid. The 27 states are present, and SP has 40.4% of the original rows, then MG 12.6%, RJ 12.1%, RS 6.2% and PR 5.8%.
- **Decision:** ignore the city text. Use only the prefix and, for the state, the most frequent state per prefix.

#### Prefix lookup (built from this table only)
**Question:** what does one row per zip prefix look like when the cleaning rules above are applied?

In [9]:
# Rules: no exact copies, only points inside Brazil, median coordinates, most frequent state, spread flag
geo_prefix = valid.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "median"),
    lng=("geolocation_lng", "median"),
    n_points=("geolocation_lat", "size"),
    state=("geolocation_state", lambda s: s.mode().iat[0]),       # most frequent state
)
geo_prefix["spread_km"] = box["spread_km"].round(1)
geo_prefix["low_quality"] = geo_prefix["spread_km"] > 50
geo_prefix["zip5"] = geo_prefix.index.astype(str).str.zfill(5)

print("lookup shape:", geo_prefix.shape, "| prefix unique:", geo_prefix.index.is_unique)
print("prefixes in the raw table   :", geo_raw["geolocation_zip_code_prefix"].nunique())
print("prefixes without a valid point (cannot be located):",
      geo_raw["geolocation_zip_code_prefix"].nunique() - len(geo_prefix))
print("low quality prefixes (spread > 50 km):", geo_prefix["low_quality"].sum())
geo_prefix.head(3).round(4)

lookup shape: (19010, 7) | prefix unique: True
prefixes in the raw table   : 19015
prefixes without a valid point (cannot be located): 5
low quality prefixes (spread > 50 km): 484


,lat,lng,n_points,state,spread_km,low_quality,zip5
geolocation_zip_code_prefix,,,,,,,
1001,-23.5500,-46.6340,11,SP,0.3,False,01001
1002,-23.5482,-46.6352,6,SP,0.6,False,01002
1003,-23.5490,-46.6353,11,SP,0.2,False,01003


- The lookup has one row per prefix and a unique index: 19,010 of the 19,015 prefixes. The 5 missing ones have no valid point.
- 484 prefixes are flagged as low quality (spread above 50 km). Their coordinates should be used with care.
- It is ready for use after the joins, where the customer and seller prefixes are matched to it.

#### Summary
**The story in short.** The geolocation table is a point cloud, not a lookup: 1,000,163 rows for 19,015 prefixes, with 26.2% exact copies, a few points outside Brazil, and prefixes whose points are hundreds of kilometres apart. The city text is unusable (accents, spelling variants, broken encoding) and some states are wrong. The reliable core is the prefix plus the median of its in-Brazil points, which gives 19,010 locatable prefixes, 484 of them with a low-quality flag.

**Data quality issues**

| # | Issue | Records | What it is | Action | Cost of the alternative |
|---|---|---|---|---|---|
| 1 | Exact copies | 261,831 rows (26.2%) | Repeated rows | Remove copies in the working table | Counts and medians are biased toward repeated points |
| 2 | Coordinates outside Brazil | 33 distinct rows, 21 prefixes | Wrong points | Ignore them in the lookup | A mean would be pulled far away |
| 3 | Prefixes with no valid point | 5 | Cannot be located | Leave empty, flag | Inventing coordinates is a guess |
| 4 | Wide spread inside one prefix | 484 over 50 km, 53 over 1,000 km | Wrong points inside Brazil | Median per prefix plus `low_quality` flag | A mean or one random point gives wrong distances |
| 5 | Zip prefix lost its leading zero | 245,733 rows (24.6%) | Stored as a number | Keep numeric for the join, add `zip5` | A text join silently matches nothing |
| 6 | Messy city text | 8,556 prefixes with several names, 19 symbol names, 2 broken encodings | Spelling variants, encoding errors | Ignore city text, use the prefix | Grouping by city splits places |
| 7 | Prefix under more than one state | 8 | Wrong state on some rows | Use the most frequent state per prefix | Taking the first row keeps a wrong state |

**Checks planned after the joins** (not done here on purpose)
- Does every customer prefix and every seller prefix exist in the lookup?
- Seller state and customer state against the state of their prefix.
- Distance between seller and customer prefixes, and its relation to delay and bad reviews.
- How many orders use a low-quality prefix.

**What can be used at order approval (for Part 2)**

| Field | Known at approval? | Note |
|---|---|---|
| Prefix coordinates (`lat`, `lng`) | Yes | Fixed reference data. Seller and customer prefixes are known at checkout. |
| `spread_km`, `low_quality` | Yes | A reliability flag for the distance feature |
| `state` from the lookup | Yes | More reliable than `seller_state` |
| City text | Not used | Messy, ignored |